In [2]:
"""
Notebook 06: Implementação de Estratégias Avançadas para MLP
Foco na Fase 1 do ADR-006: Focal Loss e Otimização via OneCycleLR com AdamW.
"""
import os
import sys

# Adiciona o src/ ao PYTHONPATH para import do config
sys.path.append(os.path.abspath(os.path.join('..')))

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader, TensorDataset

# Integração de constantes do projeto
from src.config import CONFIG

# Constantes locais do experimento
RANDOM_STATE = CONFIG.random_state
TEST_SIZE = 0.2
VAL_SIZE = 0.15
BATCH_SIZE = 256
N_EPOCHS = 300
PATIENCE = 20
N_TRIALS_OPTUNA = 20
PATH_DATA = '../notebooks/data/processed/churn_processed_advanced.csv'
EXPERIMENT_NAME = "04_PyTorch_Advanced_Loss"

# Configurações de Reproducibilidade e Device
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Usando device: {device}")

Usando device: mps


In [3]:
class FocalLoss(nn.Module):
    """
    Função de Perda Focal (Focal Loss) para Classificação Binária.

    Aborda o desbalanceamento de classes através de ponderação (alpha) e
    reduz dinamicamente o gradiente para exemplos fáceis (gamma).

    Args:
        alpha (float): Fator de ponderação para a classe minoritária (0 a 1).
            Padrão: 0.75.
        gamma (float): Fator de foco para exemplos difíceis.
            Valores maiores reduzem a perda para predições com alta confiança.
            Padrão: 2.0.
    """

    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Calcula a Focal Loss.

        Args:
            logits (torch.Tensor): Previsões cruas do modelo (antes da sigmoid).
            targets (torch.Tensor): Rótulos verdadeiros.

        Returns:
            torch.Tensor: Perda média calculada para o batch.
        """
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")

        # P_t é a probabilidade estimada do modelo para a classe alvo real
        p_t = torch.exp(-bce_loss)

        # Fator modulador: diminui para exemplos bem classificados (P_t -> 1)
        focal_weight = self.alpha * (1 - p_t) ** self.gamma

        loss = focal_weight * bce_loss
        return loss.mean()

In [4]:
# Garantir que estamos puxando as features avançadas
df = pd.read_csv(PATH_DATA)

target_col = "Churn"
X = df.drop(columns=[target_col])
y = df[target_col]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train
)

INPUT_DIM = X_train.shape[1]

In [5]:
class ChurnMLP(nn.Module):
    """
    Rede Neural Multi-Layer Perceptron (MLP) padrão para classificação tabular.
    """
    def __init__(self, input_dim: int, hidden_dims: list, dropout_rate: float = 0.3):
        super().__init__()
        layers = []
        in_dim = input_dim

        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = h_dim

        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Processa as features numéricas através da rede densa."""
        return self.network(x)

In [6]:
def train_mlp_advanced(
    model: nn.Module,
    X_tr_np: np.ndarray,
    y_tr_np: np.ndarray,
    X_val_np: np.ndarray,
    y_val_np: np.ndarray,
    loss_type: str = "bce",
    pos_weight: float = 1.0,
    focal_gamma: float = 2.0,
    focal_alpha: float = 0.75,
    n_epochs: int = 150,
    batch_size: int = 64,
    max_lr: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 20
) -> tuple:
    """
    Realiza o treinamento avançado da rede neural com Early Stopping, AdamW e OneCycleLR.
    """
    # 1. Preparação dos Datasets
    X_tr_t = torch.tensor(X_tr_np, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr_np, dtype=torch.float32).view(-1, 1)
    X_val_t = torch.tensor(X_val_np, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_np, dtype=torch.float32).view(-1, 1)

    dataset_tr = TensorDataset(X_tr_t, y_tr_t)
    loader = DataLoader(dataset_tr, batch_size=batch_size, shuffle=True)

    # 2. Definição da Loss e Otimizador
    if loss_type == "focal":
        criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma).to(device)
    else:
        pw = torch.tensor([pos_weight], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)

    # OneCycleLR (max_lr é atingido a 30% do treino, depois decai)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        steps_per_epoch=len(loader),
        epochs=n_epochs,
        pct_start=0.3
    )

    best_pr_auc = 0.0
    patience_cnt = 0
    best_state = None
    history = []

    # 3. Loop de Treinamento
    for epoch in range(1, n_epochs + 1):
        model.train()
        train_losses = []

        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()

            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

            # Step do OneCycleLR é feito A CADA BATCH
            scheduler.step()

            train_losses.append(loss.item())

        # 4. Avaliação e Early Stopping
        model.eval()
        with torch.no_grad():
            X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_probs = torch.sigmoid(val_logits).cpu().numpy()

            if np.isnan(val_probs).any():
                val_probs = np.nan_to_num(val_probs, nan=0.0)
            val_pr_auc = average_precision_score(y_val_t.cpu().numpy(), val_probs)
            val_roc_auc = roc_auc_score(y_val_t.cpu().numpy(), val_probs)

        history.append({
            "epoch": epoch,
            "train_loss": np.mean(train_losses),
            "val_loss": val_loss,
            "val_pr_auc": val_pr_auc,
            "val_roc_auc": val_roc_auc
        })

        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            patience_cnt = 0
            best_state = model.state_dict()
        else:
            patience_cnt += 1

        if patience_cnt >= patience:
            print(f"Early stopping na época {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


In [7]:
# Configuração do MLflow
MLFLOW_DB = "sqlite:///../mlflow.db"
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

def objective(trial):
    """Função objetivo para otimização Bayesiana da rede com Focal Loss."""

    # Espaço de Busca da Arquitetura
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    hidden_size_1 = trial.suggest_categorical("hidden_size_1", [32, 64, 128])
    hidden_size_2 = trial.suggest_categorical("hidden_size_2", [16, 32, 64])

    # Espaço de Busca da Topologia de Loss
    focal_gamma = trial.suggest_float("focal_gamma", 0.0, 5.0)
    focal_alpha = trial.suggest_float("focal_alpha", 0.1, 0.9)
    max_lr = trial.suggest_float("max_lr", 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    hidden_dims = [hidden_size_1, hidden_size_2]
    model = ChurnMLP(INPUT_DIM, hidden_dims, dropout_rate).to(device)

    # Treinamento
    model, history = train_mlp_advanced(
        model=model,
        X_tr_np=X_tr.values,
        y_tr_np=y_tr.values.astype(np.float32),
        X_val_np=X_val.values,
        y_val_np=y_val.values.astype(np.float32),
        loss_type="focal",
        focal_gamma=focal_gamma,
        focal_alpha=focal_alpha,
        max_lr=max_lr,
        weight_decay=weight_decay
    )

    hist_df = pd.DataFrame(history)
    return hist_df['val_pr_auc'].max()

# Instanciar e rodar o estudo (limitado a N_TRIALS_OPTUNA)
study = optuna.create_study(direction="maximize", study_name="focal_loss_tuning")
study.optimize(objective, n_trials=N_TRIALS_OPTUNA)

print(f"Melhor PR-AUC: {study.best_value}")
print(f"Melhores parâmetros: {study.best_params}")

2026/04/26 12:06:19 INFO mlflow.tracking.fluent: Experiment with name '04_PyTorch_Advanced_Loss' does not exist. Creating a new experiment.
[I 2026-04-26 12:06:19,480] A new study created in memory with name: focal_loss_tuning
[I 2026-04-26 12:06:30,837] Trial 0 finished with value: 0.671535262648516 and parameters: {'dropout_rate': 0.3949994330583828, 'hidden_size_1': 64, 'hidden_size_2': 32, 'focal_gamma': 1.9273725362870653, 'focal_alpha': 0.604062043206336, 'max_lr': 0.0007035774008345401, 'weight_decay': 0.0005593580193004215}. Best is trial 0 with value: 0.671535262648516.


Early stopping na época 61


[I 2026-04-26 12:06:36,323] Trial 1 finished with value: 0.6718669156033966 and parameters: {'dropout_rate': 0.20459181454410183, 'hidden_size_1': 64, 'hidden_size_2': 16, 'focal_gamma': 4.1152350948973995, 'focal_alpha': 0.22323758677519062, 'max_lr': 0.004335543305771928, 'weight_decay': 0.0007460231710557214}. Best is trial 1 with value: 0.6718669156033966.


Early stopping na época 35


[I 2026-04-26 12:06:41,934] Trial 2 finished with value: 0.6787699532274205 and parameters: {'dropout_rate': 0.20516687534904776, 'hidden_size_1': 64, 'hidden_size_2': 64, 'focal_gamma': 2.908507033000473, 'focal_alpha': 0.5446346596580572, 'max_lr': 0.0049313020976067224, 'weight_decay': 0.00013442795598336702}. Best is trial 2 with value: 0.6787699532274205.


...

Early stopping na época 27


[I 2026-04-26 12:08:17,208] Trial 18 finished with value: 0.6751768484777423 and parameters: {'dropout_rate': 0.28312523821096186, 'hidden_size_1': 128, 'hidden_size_2': 32, 'focal_gamma': 2.168283186287182, 'focal_alpha': 0.5104954345807349, 'max_lr': 0.007029340900504468, 'weight_decay': 0.00029900962399402337}. Best is trial 3 with value: 0.691321397349731.


Early stopping na época 40


[I 2026-04-26 12:09:54,232] Trial 19 finished with value: 0.692755610366389 and parameters: {'dropout_rate': 0.44994385411173293, 'hidden_size_1': 32, 'hidden_size_2': 64, 'focal_gamma': 0.43966925398090884, 'focal_alpha': 0.35050831069413996, 'max_lr': 0.040152668861223983, 'weight_decay': 1.659137593997329e-05}. Best is trial 19 with value: 0.692755610366389.


Early stopping na época 27
Melhor PR-AUC: 0.692755610366389
Melhores parâmetros: {'dropout_rate': 0.44994385411173293, 'hidden_size_1': 32, 'hidden_size_2': 64, 'focal_gamma': 0.43966925398090884, 'focal_alpha': 0.35050831069413996, 'max_lr': 0.040152668861223983, 'weight_decay': 1.659137593997329e-05}


In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

best_params = study.best_params

# Recriar e treinar o modelo com os melhores hiperparâmetros
best_hidden_dims = [best_params["hidden_size_1"], best_params["hidden_size_2"]]
final_model = ChurnMLP(INPUT_DIM, best_hidden_dims, best_params["dropout_rate"]).to(device)

final_model, history = train_mlp_advanced(
    model=final_model,
    X_tr_np=X_tr.values,
    y_tr_np=y_tr.values.astype(np.float32),
    X_val_np=X_val.values,
    y_val_np=y_val.values.astype(np.float32),
    loss_type="focal",
    focal_gamma=best_params["focal_gamma"],
    focal_alpha=best_params["focal_alpha"],
    max_lr=best_params["max_lr"],
    weight_decay=best_params["weight_decay"]
)

# Avaliação final no Test Set
final_model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test.values.astype(np.float32), dtype=torch.float32).view(-1, 1).to(device)

    test_logits = final_model(X_test_t)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()

    # Limiar padrão 0.5 (você pode rodar a otimização de threshold depois se necessário)
    test_preds = (test_probs >= 0.5).astype(int)
    test_pr_auc = average_precision_score(y_test, test_probs)
    test_roc_auc = roc_auc_score(y_test, test_probs)
    test_f1 = f1_score(y_test, test_preds)
    test_precision = precision_score(y_test, test_preds)
    test_recall = recall_score(y_test, test_preds)

# Registrar artefato e hiperparâmetros no MLflow
with mlflow.start_run(run_name="MLP_Focal_OneCycleLR"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        "test_pr_auc": test_pr_auc,
        "test_roc_auc": test_roc_auc,
        "test_f1": test_f1,
        "test_precision": test_precision,
        "test_recall": test_recall
    })

    # Signature input_example (Clean Code para evitar warnings)
    input_example = X_test.head(1).values.astype(np.float32)

    # 1. Mover final_model para CPU
    final_model.cpu()

    # 3. Explicitly add the signature
    signature = mlflow.models.infer_signature(
        input_example, 
        final_model(torch.tensor(input_example).cpu()).detach().numpy()
    )

    mlflow.pytorch.log_model(
        final_model,
        # 2. Replace artifact_path with name
        name="model",
        registered_model_name="MLP_Focal_OneCycleLR",
        input_example=input_example,
        signature=signature
    )

    print(f"Test PR-AUC final: {test_pr_auc:.4f}")

2026/04/26 12:10:46 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Early stopping na época 29


2026/04/26 12:10:48 INFO mlflow.models.model: Found the following environment variables used during model inference: [GEMINI_API_KEY, PERPLEXITY_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


Test PR-AUC final: 0.6530


Successfully registered model 'MLP_Focal_OneCycleLR'.
Created version '1' of model 'MLP_Focal_OneCycleLR'.


---
## Correção Metodológica: Validação Cruzada (K-Fold) na Arquitetura Avançada

Assim como ocorreu no MLP Vanilla, a arquitetura avançada sofria de *Hyperparameter Overfitting* por testar repetidas vezes o mesmo conjunto de validação (`X_val`). Para aferirmos o real poder da `FocalLoss` combinada com o `AdamW` e `OneCycleLR`, precisamos submeter o Optuna a um `StratifiedKFold` sobre o conjunto de treino inteiro. O objetivo passa a ser a maximização da **média** de PR-AUC nos Folds.

In [9]:
from sklearn.model_selection import StratifiedKFold

# Constantes K-Fold
N_SPLITS = 3
N_TRIALS_KFOLD = 15

def objective_kfold(trial):
    """
    Função objetivo do Optuna utilizando Validação Cruzada K-Fold para a Arquitetura Focal.
    O Optuna tentará otimizar os parâmetros que maximizam a média de PR-AUC dos 3 folds.
    """
    # 1. Sugestão de Hiperparâmetros (Restritos para mitigar Overfitting)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    hidden_size_1 = trial.suggest_categorical("hidden_size_1", [32, 64])
    hidden_size_2 = trial.suggest_categorical("hidden_size_2", [16, 32])
    focal_gamma = trial.suggest_float("focal_gamma", 0.0, 5.0)
    focal_alpha = trial.suggest_float("focal_alpha", 0.1, 0.9)
    max_lr = trial.suggest_float("max_lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 5e-3, log=True)

    hidden_dims = [hidden_size_1, hidden_size_2]
    
    # 2. Configurar o K-Fold no conjunto de treino original
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []
    
    X_train_np = X_train.values
    y_train_np = y_train.values.astype(np.float32)
    
    # 3. Iterar sobre cada Fold
    for train_idx, val_idx in skf.split(X_train_np, y_train_np):
        X_fold_tr, y_fold_tr = X_train_np[train_idx], y_train_np[train_idx]
        X_fold_val, y_fold_val = X_train_np[val_idx], y_train_np[val_idx]
        
        # Instanciar nova rede a cada fold
        model_fold = ChurnMLP(INPUT_DIM, hidden_dims, dropout_rate).to(device)
        
        # Treinar usando train_mlp_advanced
        model_fold, history = train_mlp_advanced(
            model=model_fold,
            X_tr_np=X_fold_tr,
            y_tr_np=y_fold_tr,
            X_val_np=X_fold_val,
            y_val_np=y_fold_val,
            loss_type="focal",
            focal_gamma=focal_gamma,
            focal_alpha=focal_alpha,
            max_lr=max_lr,
            weight_decay=weight_decay,
            n_epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            patience=PATIENCE
        )
        
        # Coletar pico de PR-AUC do fold
        hist_df = pd.DataFrame(history)
        best_fold_pr_auc = hist_df['val_pr_auc'].max()
        fold_scores.append(best_fold_pr_auc)
        
    return np.mean(fold_scores)

# Executar o Estudo
study_kfold = optuna.create_study(direction="maximize", study_name="focal_loss_kfold")
study_kfold.optimize(objective_kfold, n_trials=N_TRIALS_KFOLD)

print(f"Melhor PR-AUC Médio (K-Fold): {study_kfold.best_value:.4f}")
print("Melhores Hiperparâmetros:", study_kfold.best_params)

[I 2026-04-26 12:10:48,378] A new study created in memory with name: focal_loss_kfold


Early stopping na época 88
Early stopping na época 174


[I 2026-04-26 12:11:06,632] Trial 0 finished with value: 0.6499044977678452 and parameters: {'dropout_rate': 0.32583770233894493, 'hidden_size_1': 32, 'hidden_size_2': 32, 'focal_gamma': 1.8266440272487783, 'focal_alpha': 0.5726251847945821, 'max_lr': 0.0003783023955872453, 'weight_decay': 0.0013443376536061199}. Best is trial 0 with value: 0.6499044977678452.


Early stopping na época 151
Early stopping na época 162
Early stopping na época 98


[I 2026-04-26 12:11:21,323] Trial 1 finished with value: 0.6590763391233544 and parameters: {'dropout_rate': 0.43570941215575953, 'hidden_size_1': 32, 'hidden_size_2': 32, 'focal_gamma': 3.789333732261052, 'focal_alpha': 0.6365054776896218, 'max_lr': 0.003242674849586427, 'weight_decay': 0.0013012576310408458}. Best is trial 1 with value: 0.6590763391233544.


...

Early stopping na época 140
Early stopping na época 129
Early stopping na época 111


[I 2026-04-26 12:14:28,996] Trial 13 finished with value: 0.659726993150802 and parameters: {'dropout_rate': 0.47229556638147424, 'hidden_size_1': 64, 'hidden_size_2': 32, 'focal_gamma': 4.472579681094562, 'focal_alpha': 0.7166160792612054, 'max_lr': 0.001571639887028693, 'weight_decay': 0.0007372106574277763}. Best is trial 5 with value: 0.6626607027630795.


Early stopping na época 118
Early stopping na época 110
Early stopping na época 96


[I 2026-04-26 12:14:42,482] Trial 14 finished with value: 0.6612354372969235 and parameters: {'dropout_rate': 0.3833113743675249, 'hidden_size_1': 32, 'hidden_size_2': 16, 'focal_gamma': 1.746361019764007, 'focal_alpha': 0.8619324977674656, 'max_lr': 0.0009073092633925761, 'weight_decay': 0.0039564381879901404}. Best is trial 5 with value: 0.6626607027630795.


Early stopping na época 125
Melhor PR-AUC Médio (K-Fold): 0.6627
Melhores Hiperparâmetros: {'dropout_rate': 0.3850362197316057, 'hidden_size_1': 32, 'hidden_size_2': 16, 'focal_gamma': 3.1498786162936545, 'focal_alpha': 0.727088461640014, 'max_lr': 0.004597257944631667, 'weight_decay': 0.0002537125773816143}


In [10]:
best_params_kf = study_kfold.best_params
best_hidden_dims_kf = [best_params_kf["hidden_size_1"], best_params_kf["hidden_size_2"]]

# Instanciar modelo campeão do K-Fold
final_kfold_model = ChurnMLP(INPUT_DIM, best_hidden_dims_kf, best_params_kf["dropout_rate"]).to(device)

# Treinamento simulando hold-out com X_tr e X_val para preservar early stopping original
final_kfold_model, _ = train_mlp_advanced(
    model=final_kfold_model,
    X_tr_np=X_tr.values,
    y_tr_np=y_tr.values.astype(np.float32),
    X_val_np=X_val.values,
    y_val_np=y_val.values.astype(np.float32),
    loss_type="focal",
    focal_gamma=best_params_kf["focal_gamma"],
    focal_alpha=best_params_kf["focal_alpha"],
    max_lr=best_params_kf["max_lr"],
    weight_decay=best_params_kf["weight_decay"],
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE
)

# Avaliação rigorosa no Test Set (Hold-out Cego)
final_kfold_model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test.values).to(device)
    y_test_t = torch.FloatTensor(y_test.values.astype(np.float32)).to(device)
    
    test_logits_kf = final_kfold_model(X_test_t).squeeze()
    test_probs_kf = torch.sigmoid(test_logits_kf).cpu().numpy()
    
    test_preds_kf = (test_probs_kf >= 0.5).astype(int)
    test_pr_auc_kf = average_precision_score(y_test.values, test_probs_kf)
    test_roc_auc_kf = roc_auc_score(y_test.values, test_probs_kf)
    test_f1_kf = f1_score(y_test.values, test_preds_kf)
    test_precision_kf = precision_score(y_test.values, test_preds_kf)
    test_recall_kf = recall_score(y_test.values, test_preds_kf)

print(f"Test PR-AUC do Modelo Vencedor Avançado (K-Fold): {test_pr_auc_kf:.4f}")

# Registro MLOps no MLflow
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="MLP_Advanced_KFold"):
    mlflow.log_params(best_params_kf)
    mlflow.log_metrics({
        "test_pr_auc": test_pr_auc_kf,
        "test_roc_auc": test_roc_auc_kf,
        "test_f1": test_f1_kf,
        "test_precision": test_precision_kf,
        "test_recall": test_recall_kf
    })
    
    # Move para CPU para evitar Tensor Error RuntimeError('Tensor for argument input is on cpu but expected on mps')
    final_kfold_model.cpu()
    
    input_sample = X_test.head(1).values.astype(np.float32)
    output_sample = final_kfold_model(torch.tensor(input_sample)).detach().numpy()
    sig_kfold = infer_signature(input_sample, output_sample)
    
    mlflow.pytorch.log_model(
        final_kfold_model,
        name="model",
        registered_model_name="MLP_Focal_KFold",
        signature=sig_kfold
    )

2026/04/26 12:14:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Early stopping na época 48
Test PR-AUC do Modelo Vencedor Avançado (K-Fold): 0.6506


Successfully registered model 'MLP_Focal_KFold'.
Created version '1' of model 'MLP_Focal_KFold'.
